# Tumour-marker PCA and silhouette analysis for UNKNOWNS-69

Corrected notebook for the EDTA Cleaner case study. This version uses the 69-sample UNKNOWNS dataset and excludes the three non-authoritative CRC171 samples (`CRC171`, `CRC171/2`, `CRC171/3`).

The notebook reports two silhouette metrics:

- **PC1-PC2 silhouette**: calculated from the shared two-dimensional PCA coordinates used in Fig. 6.
- **Full-space silhouette**: calculated from the full standardised tumour-marker feature space and used for the quantitative comparison in the main text and Supplementary Table S13.


## Case-study scope

This notebook contains the tumour-marker case-study analysis for the UNKNOWNS-69 subset derived from the previously published colorectal cancer cohort (Slyskova et al., 2015). It replaces the brief case-study preview that was previously included at the end of the SVM decision-boundary notebook.

The analysis uses three fixed input tables stored in data/:

- tumour_marker_case_study_UNKNOWNS_69.csv: all 69 tumour-marker samples.
- tumour_marker_case_study_EDTA_Cleaner_FNR10_filtered.csv: 45 samples retained after EDTA Cleaner FNR10 filtering.
- tumour_marker_case_study_RIN_filtered.csv: 59 samples retained after RIN filtering.

The figure generated below uses the shared PC1-PC2 PCA projection for visual comparison, whereas the full-space silhouette scores provide the main quantitative values for the manuscript and Supplementary Table S13.



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

try:
    from scipy.stats import gaussian_kde
    SCIPY_AVAILABLE = True
except ModuleNotFoundError:
    SCIPY_AVAILABLE = False

print('scipy available:', SCIPY_AVAILABLE)


In [ ]:
cwd = Path().cwd()
ROOT = cwd
while ROOT != ROOT.parent and not (ROOT / 'README.md').exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data'
NB_OUTPUT = ROOT / 'notebooks' / 'outputs_from_notebooks'
NB_OUTPUT.mkdir(parents=True, exist_ok=True)

UNKNOWN69_PATH = DATA_DIR / 'tumour_marker_case_study_UNKNOWNS_69.csv'
UNKNOWN69_EDTA_PATH = DATA_DIR / 'tumour_marker_case_study_EDTA_Cleaner_FNR10_filtered.csv'
UNKNOWN69_RIN_PATH = DATA_DIR / 'tumour_marker_case_study_RIN_filtered.csv'

for label, path in [
    ('UNKNOWNS-69 tumour markers', UNKNOWN69_PATH),
    ('EDTA Cleaner-filtered subset', UNKNOWN69_EDTA_PATH),
    ('RIN-filtered subset', UNKNOWN69_RIN_PATH),
]:
    print(f'{label}: {path} Exists? {path.exists()}')


In [ ]:
marker_cols = [
    'LIG3', 'RPA3', 'CDK7', 'DDB2', 'ERCC5',
    'ERCC1', 'NBN', 'ATR', 'CHEK1', 'CHEK2', 'TP53'
]
class_names = {0: 'Healthy', 1: 'Diseased'}
removed_non_authoritative = {'CRC171', 'CRC171/2', 'CRC171/3'}

all_df = pd.read_csv(UNKNOWN69_PATH, index_col=0)
edta_df = pd.read_csv(UNKNOWN69_EDTA_PATH, index_col=0)
rin_df = pd.read_csv(UNKNOWN69_RIN_PATH, index_col=0)
for df in [all_df, edta_df, rin_df]:
    df.index = df.index.astype(str)

# Fail loudly if an old 72-sample file is loaded by mistake.
for df_name, df, expected_n in [('all_df', all_df, 69), ('edta_df', edta_df, 45), ('rin_df', rin_df, 59)]:
    assert len(df) == expected_n, f'{df_name}: expected {expected_n}, got {len(df)}'
    assert not any(sample in df.index for sample in removed_non_authoritative), f'{df_name}: CRC171 samples still present'
    assert all(col in df.columns for col in marker_cols + ['treatment']), f'{df_name}: missing required columns'
    assert df.index.is_unique, f'{df_name}: duplicate sample IDs'

# Confirm filtered subset values match the same samples in the unfiltered dataset.
for label, sub_df in [('EDTA Cleaner-filtered', edta_df), ('RIN-filtered', rin_df)]:
    common = sub_df.index.intersection(all_df.index)
    max_abs_diff = (sub_df.loc[common, marker_cols] - all_df.loc[common, marker_cols]).abs().to_numpy().max()
    assert max_abs_diff == 0, f'{label}: marker values differ from all_df'

summary_counts = pd.DataFrame([
    {'Dataset': 'Unfiltered', 'n': len(all_df), **all_df['treatment'].map(class_names).value_counts().to_dict()},
    {'Dataset': 'EDTA Cleaner-filtered', 'n': len(edta_df), **edta_df['treatment'].map(class_names).value_counts().to_dict()},
    {'Dataset': 'RIN-filtered', 'n': len(rin_df), **rin_df['treatment'].map(class_names).value_counts().to_dict()},
]).fillna(0)
summary_counts


In [ ]:
# Standardise using the unfiltered UNKNOWNS-69 dataset as reference.
X_all_raw = all_df[marker_cols].to_numpy(dtype=float)
X_edta_raw = edta_df[marker_cols].to_numpy(dtype=float)
X_rin_raw = rin_df[marker_cols].to_numpy(dtype=float)
y_all = all_df['treatment'].to_numpy()
y_edta = edta_df['treatment'].to_numpy()
y_rin = rin_df['treatment'].to_numpy()

scaler = StandardScaler().fit(X_all_raw)
X_all = scaler.transform(X_all_raw)
X_edta = scaler.transform(X_edta_raw)
X_rin = scaler.transform(X_rin_raw)
print('Scaled shapes:', X_all.shape, X_edta.shape, X_rin.shape)


In [ ]:
# Full-space silhouette scores: quantitative metric reported in the main text and S13.
full_space_scores = {
    'Unfiltered': float(silhouette_score(X_all, y_all)),
    'EDTA Cleaner-filtered': float(silhouette_score(X_edta, y_edta)),
    'RIN-filtered': float(silhouette_score(X_rin, y_rin)),
}
for key, value in full_space_scores.items():
    print(f'{key}: {value:.6f} ({value:.3f})')


In [ ]:
# Shared PCA projection for Fig. 6: fit PCA on unfiltered UNKNOWNS-69 only.
pca2 = PCA(n_components=2, random_state=42).fit(X_all)
X2_all = pca2.transform(X_all)
X2_edta = pca2.transform(X_edta)
X2_rin = pca2.transform(X_rin)

# PCA signs are arbitrary. The corrected unscaled input data flip both PCs
# relative to the previous figure, so rotate the displayed coordinates by 180?
# to keep the final visual orientation unchanged.
pca_display_orientation = np.array([-1.0, -1.0])
X2_all = X2_all * pca_display_orientation
X2_edta = X2_edta * pca_display_orientation
X2_rin = X2_rin * pca_display_orientation

explained_variance_ratio = pca2.explained_variance_ratio_

pc1_pc2_scores = {
    'Unfiltered': float(silhouette_score(X2_all, y_all)),
    'EDTA Cleaner-filtered': float(silhouette_score(X2_edta, y_edta)),
    'RIN-filtered': float(silhouette_score(X2_rin, y_rin)),
}
print('Explained variance ratio:', explained_variance_ratio, 'sum=', explained_variance_ratio.sum())
for key, value in pc1_pc2_scores.items():
    print(f'{key}: {value:.6f} ({value:.3f})')


In [ ]:
silhouette_summary = pd.DataFrame([
    {'Dataset': 'Unfiltered', 'n total': len(all_df), 'Healthy': int((y_all == 0).sum()), 'Diseased': int((y_all == 1).sum()), 'PC1-PC2 silhouette': pc1_pc2_scores['Unfiltered'], 'Full-space silhouette': full_space_scores['Unfiltered']},
    {'Dataset': 'EDTA Cleaner-filtered', 'n total': len(edta_df), 'Healthy': int((y_edta == 0).sum()), 'Diseased': int((y_edta == 1).sum()), 'PC1-PC2 silhouette': pc1_pc2_scores['EDTA Cleaner-filtered'], 'Full-space silhouette': full_space_scores['EDTA Cleaner-filtered']},
    {'Dataset': 'RIN-filtered', 'n total': len(rin_df), 'Healthy': int((y_rin == 0).sum()), 'Diseased': int((y_rin == 1).sum()), 'PC1-PC2 silhouette': pc1_pc2_scores['RIN-filtered'], 'Full-space silhouette': full_space_scores['RIN-filtered']},
])
silhouette_summary.round(3)


In [ ]:
# Stratified random-removal nulls in the full standardised tumour-marker feature space.
def stratified_random_removal_null(filtered_df, observed_silhouette, n_iter=10000, seed=42):
    kept = all_df.index.isin(filtered_df.index)
    removed_by_class = {int(cls): int(np.sum((y_all == cls) & (~kept))) for cls in np.unique(y_all)}
    rng = np.random.default_rng(seed)
    null_scores = np.empty(n_iter)
    for i in range(n_iter):
        keep_mask = np.ones(len(y_all), dtype=bool)
        for cls, n_remove in removed_by_class.items():
            cls_idx = np.flatnonzero(y_all == cls)
            remove_idx = rng.choice(cls_idx, size=n_remove, replace=False)
            keep_mask[remove_idx] = False
        null_scores[i] = silhouette_score(X_all[keep_mask], y_all[keep_mask])
    percentile = float(np.mean(null_scores <= observed_silhouette) * 100)
    empirical_p = float((np.sum(null_scores >= observed_silhouette) + 1) / (n_iter + 1))
    return removed_by_class, null_scores, percentile, empirical_p

removed_edta, null_edta, pct_edta, p_edta = stratified_random_removal_null(edta_df, full_space_scores['EDTA Cleaner-filtered'], n_iter=10000, seed=42)
removed_rin, null_rin, pct_rin, p_rin = stratified_random_removal_null(rin_df, full_space_scores['RIN-filtered'], n_iter=10000, seed=43)

random_removal_summary = pd.DataFrame([
    {'Filter': 'EDTA Cleaner', 'Removed total': sum(removed_edta.values()), 'Removed Healthy': removed_edta.get(0, 0), 'Removed Diseased': removed_edta.get(1, 0), 'Observed full-space silhouette': full_space_scores['EDTA Cleaner-filtered'], 'Null percentile': pct_edta, 'Empirical one-sided p value': p_edta},
    {'Filter': 'RIN', 'Removed total': sum(removed_rin.values()), 'Removed Healthy': removed_rin.get(0, 0), 'Removed Diseased': removed_rin.get(1, 0), 'Observed full-space silhouette': full_space_scores['RIN-filtered'], 'Null percentile': pct_rin, 'Empirical one-sided p value': p_rin},
])
random_removal_summary


In [ ]:
# Removed-sample overlap check.
edta_removed = set(all_df.index) - set(edta_df.index)
rin_removed = set(all_df.index) - set(rin_df.index)
print('EDTA Cleaner removed:', len(edta_removed))
print('RIN removed:', len(rin_removed))
print('Removed by both:', len(edta_removed & rin_removed), sorted(edta_removed & rin_removed))


In [ ]:
# Figure 6-style shared-PCA plot. Figure annotations use PC1-PC2 silhouette values.
COLORS = {'Healthy': '#2CB2BA', 'Diseased': '#FFBF00'}
plt.rcParams.update({'font.size': 8, 'axes.titlesize': 9, 'axes.labelsize': 8, 'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7})
fig, axes = plt.subplots(1, 3, figsize=(12.96, 4.86), sharex=True, sharey=True)
datasets = [(X2_all, y_all, 'All samples', pc1_pc2_scores['Unfiltered']), (X2_edta, y_edta, 'EDTA Cleaner-filtered', pc1_pc2_scores['EDTA Cleaner-filtered']), (X2_rin, y_rin, 'RIN >= 8.5', pc1_pc2_scores['RIN-filtered'])]
combined = np.vstack([X2_all, X2_edta, X2_rin])
x_min, x_max = combined[:, 0].min() - 0.5, combined[:, 0].max() + 0.5
y_min, y_max = combined[:, 1].min() - 0.5, combined[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.vstack([xx.ravel(), yy.ravel()])
for ax, (X_pca, y_lab, title, sil_value) in zip(axes, datasets):
    if SCIPY_AVAILABLE:
        kde_healthy = gaussian_kde(X_pca[y_lab == 0].T)
        kde_diseased = gaussian_kde(X_pca[y_lab == 1].T)
        Z = (kde_diseased(grid) - kde_healthy(grid)).reshape(xx.shape)
        ax.contourf(xx, yy, Z, levels=40, cmap='RdBu_r', alpha=0.30)
    for cls, name in [(0, 'Healthy'), (1, 'Diseased')]:
        idx = y_lab == cls
        ax.scatter(X_pca[idx, 0], X_pca[idx, 1], color=COLORS[name], edgecolors='k', linewidths=0.6, s=35, alpha=0.85, label=name)
    ax.text(0.98, 0.02, f'Silhouette = {sil_value:.2f}', transform=ax.transAxes, ha='right', va='bottom', fontsize=7)
    ax.set_title(title)
    ax.set_xlabel(f'PC1 ({explained_variance_ratio[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({explained_variance_ratio[1]*100:.1f}%)')
    ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
axes[0].legend(handles=[mpatches.Patch(color=COLORS['Healthy'], label='Healthy'), mpatches.Patch(color=COLORS['Diseased'], label='Diseased')], loc='lower left', frameon=False)
for label, ax in zip(['a', 'b', 'c'], axes):
    ax.text(0.0, 1.10, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom')
fig.tight_layout(rect=(0, 0.10, 1, 1))
out_base = NB_OUTPUT / 'FIGURE6_UNKNOWNS69_SHARED_PCA'
fig.savefig(str(out_base) + '.png', dpi=300, bbox_inches='tight')
fig.savefig(str(out_base) + '.pdf', bbox_inches='tight')
fig.savefig(str(out_base) + '.svg', bbox_inches='tight')
plt.show()
print('Saved:', out_base)


In [ ]:
# Export summary tables for Supplementary Table S13 cross-checking.
summary_dir = NB_OUTPUT / 'S13_case_study_outputs'
summary_dir.mkdir(parents=True, exist_ok=True)
silhouette_summary.to_csv(summary_dir / 'silhouette_scores_UNKNOWNS69.csv', index=False)
random_removal_summary.to_csv(summary_dir / 'random_removal_UNKNOWNS69.csv', index=False)
all_df.to_csv(summary_dir / 'tumour_marker_data_UNKNOWNS69.csv')
print('Exported summary files to:', summary_dir)
